## Objective

This notebook evaluates the historical return and risk characteristics of the Aggressive Growth, Balanced Growth, and Resilient Growth portfolios constructed in Notebook 2.

The analysis compares total and annualized returns, volatility, maximum drawdown, Value at Risk (VaR), Expected Shortfall (ES), and risk-adjusted performance. Together, these measures provide a more complete view of the trade-off between growth and downside risk across the three strategic allocations.

## Load Portfolio Returns


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load portfolio daily returns from Notebook 2
portfolio_returns = pd.read_csv(
    "portfolio_returns.csv",
    index_col=0,
    parse_dates=True
)

# Validate dataset
if portfolio_returns.isna().any().any():
    raise ValueError("Missing values detected in portfolio returns.")

print(
    f"Sample period: "
    f"{portfolio_returns.index.min().date()} to "
    f"{portfolio_returns.index.max().date()}"
)

print(f"Trading-day observations: {len(portfolio_returns):,}")

display(portfolio_returns.head())

Sample period: 2007-05-31 to 2026-09-18
Trading-day observations: 4,857


,Aggressive,Balanced,Resilient
Date,,,
2007-05-31,0.002857,0.002203,0.001999
2007-06-01,0.007015,0.005458,0.003460
2007-06-04,0.001120,0.000994,0.001095
2007-06-05,-0.004741,-0.004004,-0.003120
2007-06-06,-0.009978,-0.007555,-0.004277


## Performance Metrics

This section evaluates the historical performance of the three portfolios using cumulative and annualized returns. These measures show how much each strategy grew over the sample before considering the amount of risk taken to achieve that growth.

## Cumulative Return

Cumulative return measures the total percentage gain or loss generated over the full historical sample. Answers the question, "How much did this portoflio grow from beginning of the anlysis through ending date?". It provides a straightforward comparison of long-term portfolio growth but does not account for the volatility or downside risk experienced along the way.


### Calculation

In [2]:
# Calculate cumulative return over the full sample period
cumulative_returns = (1 + portfolio_returns).prod() - 1

# Display as percentages
cumulative_returns_df = (
    cumulative_returns
    .to_frame(name="Cumulative Return")
    .style
    .format("{:.2%}")
)

cumulative_returns_df

,Cumulative Return
Aggressive,379.02%
Balanced,298.11%
Resilient,235.59%


### Interpretation

The Aggressive Growth portfolio generated the greatest cumulative growth over the historical sample, followed by Balanced Growth and Resilient Growth. This reflects the greater equity exposure of the Aggressive strategy during a period in which equities contributed substantially to long-term portfolio growth.

Cumulative return alone does not indicate whether the additional growth justified the additional risk, so the following measures evaluate the return and downside characteristics of each strategy more directly.

## Annualized Return (Compound Annual Growth Rate, CAGR)

Annualized return expresses the portfolio's compounded growth as an average annual rate, making the long-term performance of the three strategies easier to compare.

CAGR summarizes the beginning and ending portfolio values into a single annual growth rate. It does not describe the path taken between those points, so periods of volatility and large losses can be hidden by the final annualized figure.


### Calculation

In [3]:
# Number of trading days
num_days = len(portfolio_returns)

# Approximate number of years
num_years = num_days / 252

# Calculate annualized returns
annualized_return = (1 + cumulative_returns) ** (1 / num_years) - 1

annualized_return_df = (
    annualized_return
    .to_frame(name="Annualized Return")
    .style
    .format("{:.2%}")
)

annualized_return_df

,Annualized Return
Aggressive,8.47%
Balanced,7.43%
Resilient,6.48%


### Interpretation

The annualized results preserve the same ordering as cumulative return: Aggressive Growth produced the highest long-term growth rate, Balanced Growth remained between the two strategies, and Resilient Growth produced the lowest. These results are consistent with the portfolios' strategic asset allocations, as higher allocations to equity assets generally produced greater long-term returns over the sample period.

The difference in return is only one side of the portfolio decision. The next section evaluates how much variability and downside risk accompanied those results.


## Risk Metrics

This section evaluates the risk characteristics of the Aggressive, Balanced, and Resilient portfolios. While the previous section measured historical performance, the following metrics quantify the uncertainty and downside risk associated with each portfolio. These measures provide a more complete assessment of portfolio performance by considering both return and risk.


### Annualized Volatility

Annualized volatility measures the historical variability of daily portfolio returns on an annualized basis. Higher volatility indicates that returns fluctuated more widely over the sample, while lower volatility indicates a more stable historical return pattern.

Volatility captures both positive and negative movements, so it should be considered alongside measures that focus specifically on downside risk.

### Calculation

In [4]:
# Calculate annualized volatility
annualized_volatility = portfolio_returns.std() * np.sqrt(252)

annualized_volatility_df = (
    annualized_volatility
    .to_frame(name="Annualized Volatility")
    .style
    .format("{:.2%}")
)

annualized_volatility_df

,Annualized Volatility
Aggressive,17.43%
Balanced,13.56%
Resilient,9.00%


### Interpretation
Aggressive Growth exhibited the greatest historical return variability, while Resilient Growth had the lowest volatility and Balanced Growth remained between the two. The pattern is consistent with the portfolios' progressively lower equity exposure and larger defensive allocations.

Volatility does not distinguish favorable gains from harmful losses, so the analysis next focuses directly on downside outcomes.

## Maximum Drawdown

Maximum Drawdown measures the largest historical decline from a portfolio peak to a subsequent trough. It provides an intuitive measure of how severe an investor's loss could have become during the worst sustained decline in the sample.

Unlike volatility, Maximum Drawdown focuses directly on downside experience, although it captures only the single worst historical drawdown and does not indicate how frequently large losses occurred.

### Calculation

In [5]:
# Calculate cumulative portfolio growth
cumulative_growth = (1 + portfolio_returns).cumprod()

# Calculate running maximum
running_max = cumulative_growth.cummax()

# Calculate drawdowns
drawdown = (cumulative_growth - running_max) / running_max

# Maximum Drawdown
max_drawdown = drawdown.min()

# Display results
max_drawdown_df = (
    max_drawdown
    .to_frame(name="Maximum Drawdown")
    .style
    .format("{:.2%}")
)

max_drawdown_df

,Maximum Drawdown
Aggressive,-50.09%
Balanced,-39.49%
Resilient,-25.77%


### Interpretation 

Maximum Drawdown shows a clear difference in downside severity across the three strategies. Aggressive Growth experienced the deepest peak-to-trough decline, Balanced Growth experienced a smaller decline, and Resilient Growth provided the strongest protection during the worst historical drawdown.

The result reinforces the trade-off already visible in the return and volatility measures: the portfolios with greater long-term growth exposure also experienced larger losses during severe market declines.

## Value at Risk (VaR)
Historical Value at Risk estimates a loss threshold from the observed return distribution. A one-day 95% Historical VaR represents the daily loss level that was exceeded on approximately 5% of historical trading days.

VaR provides a standardized way to compare short-horizon downside risk across the portfolios, but it does not describe how severe losses become after the threshold is exceeded.

### Calculation

In [6]:
# One-day Historical Value at Risk (95%)
historical_var = -portfolio_returns.quantile(0.05)

historical_var_df = (
    historical_var
    .to_frame(name="Historical VaR (95%)")
    .style
    .format("{:.2%}")
)

historical_var_df

,Historical VaR (95%)
Aggressive,1.59%
Balanced,1.23%
Resilient,0.81%


### Interpretation

The Aggressive portfolio has the largest one-day Historical VaR, indicating the greatest exposure to large daily losses. Balanced Growth has an intermediate loss threshold, while Resilient Growth has the smallest.

The measure identifies where the historical tail begins but not the severity of losses within that tail. Expected Shortfall is therefore used next to evaluate what happened on the worst trading days.

## Expected Shortfall
Expected Shortfall (ES), also called Conditional Value at Risk (CVaR), measures the average loss during the worst 5% of historical trading days.

Where VaR identifies the tail-loss threshold, Expected Shortfall describes the severity of losses after that threshold has been crossed. Because it focuses on the tail of the return distribution, ES provides additional information about extreme downside risk.


### Calculation

In [7]:
# Historical Expected Shortfall (95%)
historical_es = -portfolio_returns.apply(
    lambda x: x[x <= x.quantile(0.05)].mean()
)

historical_es_df = (
    historical_es
    .to_frame(name="Historical Expected Shortfall (95%)")
    .style
    .format("{:.2%}")
)

historical_es_df

,Historical Expected Shortfall (95%)
Aggressive,2.66%
Balanced,2.04%
Resilient,1.32%


### Interpretation

Expected Shortfall follows the same general risk ordering as VaR. Aggressive Growth experienced the most severe average losses within the historical tail, while Resilient Growth experienced the smallest and Balanced Growth remained between the two.

The comparison shows that the difference between the portfolios extends beyond ordinary volatility: the more defensive allocations also reduced the average severity of the worst historical daily losses.

Expected Shortfall remains dependent on the observed sample and cannot capture extreme events that have not occurred historically.

## Sharpe Ratio

The Sharpe Ratio compares portfolio excess return with the volatility required to generate that return. For this analysis, BIL is used as a historical proxy for the short-term risk-free return rather than applying a single constant risk-free rate across the full sample.

Because BIL is an investable Treasury-bill ETF rather than a theoretical risk-free asset, this remains an approximation. However, it allows the risk-free benchmark to change over time and is consistent with the short-term Treasury proxy already used elsewhere in the project.


### Calculation

In [8]:
# Load asset returns to obtain the short-term Treasury proxy
asset_returns = pd.read_csv(
    "returns_daily.csv",
    index_col=0,
    parse_dates=True
)

# Use BIL as a historical short-term Treasury proxy
risk_free_returns = asset_returns["BIL"].reindex(
    portfolio_returns.index
)

# Align portfolio and risk-free returns
sharpe_data = portfolio_returns.copy()
sharpe_data["Risk_Free"] = risk_free_returns
sharpe_data = sharpe_data.dropna()

# Calculate daily excess returns
excess_returns = sharpe_data[
    portfolio_returns.columns
].sub(
    sharpe_data["Risk_Free"],
    axis=0
)

# Annualized Sharpe Ratio
sharpe_ratio = (
    excess_returns.mean()
    / excess_returns.std()
    * np.sqrt(252)
)

display(
    sharpe_ratio
    .to_frame(name="Sharpe Ratio")
    .style
    .format("{:.2f}")
)

,Sharpe Ratio
Aggressive,0.47
Balanced,0.49
Resilient,0.59


### Interpretation
The Resilient Growth portfolio achieved the highest Sharpe Ratio, followed by Balanced Growth and Aggressive Growth. This indicates that, over the historical sample, the Resilient portfolio generated the greatest excess return relative to the amount of volatility it experienced.

Although Aggressive Growth produced the highest absolute return, its substantially greater volatility reduced its return per unit of risk. In contrast, the lower volatility of the Resilient portfolio more than offset its lower return when evaluated on a risk-adjusted basis.

The Sharpe Ratio therefore provides a different perspective from the raw return measures: Aggressive Growth produced the greatest historical growth, while Resilient Growth produced the strongest historical return relative to total volatility. The result should still be interpreted alongside downside-risk measures because the Sharpe Ratio treats both positive and negative volatility as risk.


## Summary 

In [9]:
# Create summary table
summary_table = pd.DataFrame({
    "Cumulative Return": cumulative_returns,
    "Annualized Return": annualized_return,
    "Annualized Volatility": annualized_volatility,
    "Maximum Drawdown": max_drawdown,
    "Historical VaR (95%)": historical_var,
    "Historical Expected Shortfall (95%)": historical_es,
    "Sharpe Ratio": sharpe_ratio
})
# Rename Index
summary_table.index = [
    "Aggressive Growth",
    "Balanced Growth",
    "Resilient Growth"
]
#Convert index to portfolio column 
summary_table = summary_table.reset_index()
summary_table.rename(columns={"index": "Portfolio"}, inplace=True)

#Display formatted table
summary_table.style.hide(axis="index").format({
    "Cumulative Return": "{:.2%}",
    "Annualized Return": "{:.2%}",
    "Annualized Volatility": "{:.2%}",
    "Maximum Drawdown": "{:.2%}",
    "Historical VaR (95%)": "{:.2%}",
    "Historical Expected Shortfall (95%)": "{:.2%}",
    "Sharpe Ratio": "{:.2f}"
}).set_caption(
    "Portfolio Risk and Performance Summary"
)

Portfolio,Cumulative Return,Annualized Return,Annualized Volatility,Maximum Drawdown,Historical VaR (95%),Historical Expected Shortfall (95%),Sharpe Ratio
Aggressive Growth,379.02%,8.47%,17.43%,-50.09%,1.59%,2.66%,0.47
Balanced Growth,298.11%,7.43%,13.56%,-39.49%,1.23%,2.04%,0.49
Resilient Growth,235.59%,6.48%,9.00%,-25.77%,0.81%,1.32%,0.59


### Overall Summary 
The historical results show a clear trade-off across the three strategic allocations. Aggressive Growth produced the strongest cumulative and annualized returns but also experienced the greatest volatility, maximum drawdown, and tail-loss exposure. Resilient Growth generated lower absolute returns but provided substantially greater downside protection, while Balanced Growth remained between the two across most measures.

The risk-adjusted results add an important distinction. Resilient Growth achieved the highest Sharpe Ratio, indicating that its lower historical return was accompanied by a sufficiently large reduction in volatility to produce the strongest excess return per unit of total risk. Balanced Growth ranked second, while Aggressive Growth produced the lowest Sharpe Ratio despite having the highest absolute return.

Taken together, the results show that the portfolio with the highest return was not necessarily the most efficient on a risk-adjusted basis. No single metric determines which strategy is preferable, however, because each portfolio is designed to serve a different risk objective. The following notebooks extend the analysis by examining how the portfolios behaved during historical crises and under hypothetical stress conditions.

In [10]:
summary_table.to_csv("portfolio_summary_metrics.csv", index = False)